In [1]:
import geopandas as gpd
import gcsfs
import pandas as pd

from google.cloud import bigquery

from shared_vars import RAW_GCS, INTERMED_GCS, AGENCY_TO_GTFS_NAME_DICT, RAW_DATA_YAML
from ridership_utils import bq_utils, utils

In [2]:
SCHEDULE_FEEDS_FILENAME = "schedule_feeds"

feeds_df = pd.read_parquet(
    f"{INTERMED_GCS}{SCHEDULE_FEEDS_FILENAME}.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

In [3]:
test_operator = "Bay Area 511 BART Schedule"

In [4]:
feeds_df[feeds_df.schedule_name==test_operator].head(2)

,feed_key,schedule_name,service_date_start,service_date_end
3810,3d1e24aeee75adb17662d9d8cb2ec350,Bay Area 511 BART Schedule,2026-05-28,2026-05-28
3811,bfb2078783000a9e17d3bdeb0a281f4f,Bay Area 511 BART Schedule,2026-04-24,2026-05-27


In [5]:
list_of_operators = list(RAW_DATA_YAML.keys())

operators_with_ridership = pd.concat([
    pd.read_parquet(
        f"{RAW_GCS}{agency_name}/ridership_round1.parquet",
        columns = ["schedule_name", "start_date", "end_date"],
        filesystem = gcsfs.GCSFileSystem()
    ) for agency_name in list_of_operators
    ], axis=0, ignore_index=True
).drop_duplicates().reset_index(drop=True)

ridership_start = operators_with_ridership[
    operators_with_ridership.schedule_name==test_operator
].start_date.min()

ridership_end = operators_with_ridership[
    operators_with_ridership.schedule_name==test_operator
].end_date.max()

In [6]:
ridership_start, ridership_end

(Timestamp('2024-10-01 00:00:00'), Timestamp('2025-09-30 00:00:00'))

In [7]:
test_feeds_df = feeds_df[
    (feeds_df.schedule_name==test_operator) 
].sort_values(
    ["service_date_start", "service_date_end"]
).reset_index(drop=True)

test_feeds_df

,feed_key,schedule_name,service_date_start,service_date_end
0,d80a42d70891b7549640562e5be47757,Bay Area 511 BART Schedule,2023-01-01,2023-01-11
1,73cdc9157518a7ee1a30809d40a3858a,Bay Area 511 BART Schedule,2023-01-12,2023-01-25
2,59b6a595ab2fea442b8000cd97f372bc,Bay Area 511 BART Schedule,2023-01-26,2023-02-13
3,cacb113abcfffee16b7ffd0fc3be77b0,Bay Area 511 BART Schedule,2023-02-14,2023-03-01
4,3e1d39f4d0919bc8aca8d5f99bfea726,Bay Area 511 BART Schedule,2023-03-02,2023-03-02
...,...,...,...,...
72,b4dda97100acf38a587f7f0fb447f6a9,Bay Area 511 BART Schedule,2026-01-01,2026-01-02
73,4d1c677d47f3c48a4323f506927383d3,Bay Area 511 BART Schedule,2026-01-03,2026-03-31
74,1aa4230f065e18b82c78ae8b0aabaf19,Bay Area 511 BART Schedule,2026-04-01,2026-04-23
75,bfb2078783000a9e17d3bdeb0a281f4f,Bay Area 511 BART Schedule,2026-04-24,2026-05-27


In [8]:
# this condition is grabbing way too much still
test_feeds_df[
    (
        (test_feeds_df.service_date_start <= ridership_start) &
        (test_feeds_df.service_date_end <= ridership_end)
    ) | 
    (
        (test_feeds_df.service_date_end >= ridership_end) &
        (test_feeds_df.service_date_start >= ridership_start)
    )
]

,feed_key,schedule_name,service_date_start,service_date_end
0,d80a42d70891b7549640562e5be47757,Bay Area 511 BART Schedule,2023-01-01,2023-01-11
1,73cdc9157518a7ee1a30809d40a3858a,Bay Area 511 BART Schedule,2023-01-12,2023-01-25
2,59b6a595ab2fea442b8000cd97f372bc,Bay Area 511 BART Schedule,2023-01-26,2023-02-13
3,cacb113abcfffee16b7ffd0fc3be77b0,Bay Area 511 BART Schedule,2023-02-14,2023-03-01
4,3e1d39f4d0919bc8aca8d5f99bfea726,Bay Area 511 BART Schedule,2023-03-02,2023-03-02
5,66b32bcbcd47ced39f98c648b038914c,Bay Area 511 BART Schedule,2023-03-03,2023-03-10
6,2083403c90fd9bd77ccb235906093136,Bay Area 511 BART Schedule,2023-03-11,2023-03-17
7,7cf2ecc1c5ff968d5acaacec0814c1c9,Bay Area 511 BART Schedule,2023-03-18,2023-03-31
8,19ce0483d15fbbec6aa8b9e27883c95e,Bay Area 511 BART Schedule,2023-04-01,2023-05-04
9,23b6c8cb61db6a290b4d554f36fe7d0f,Bay Area 511 BART Schedule,2023-05-05,2023-05-16
